# RoadSight-PK: Google Colab GPU Training & Evaluation Pipeline

**Project Repository:** [https://github.com/maryamyaseen0209/RoadSight-PK](https://github.com/maryamyaseen0209/RoadSight-PK)

This notebook trains and evaluates the complete **RoadSight-PK** pipeline on Google Colab GPU:
- **Experiment 1:** International Baseline Training (RDD2022)
- **Experiment 2:** Cross-Country Generalization (Leave-One-Country-Out / LOCO)
- **Experiment 3:** Zero-Shot Pakistani Road Assessment
- **Experiment 4:** Local Pakistani Adaptation (10%, 25%, 50%, 100% Subsets)
- **Experiment 5:** Speed & Latency Benchmarks (FPS, Latency, Pareto Frontier)
- **Experiment 6:** Error Analysis & Visualization Generation

> **Ensure GPU is enabled:** `Runtime` -> `Change runtime type` -> Select `T4 GPU` or `A100 GPU`.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone RoadSight-PK Repository
!git clone https://github.com/maryamyaseen0209/RoadSight-PK.git
%cd RoadSight-PK
!pip install -q ultralytics torchvision torchmetrics matplotlib pyyaml pillow

In [ ]:
# 3. Set Up Directories and Environment
import os
import sys
from pathlib import Path
sys.path.append(os.getcwd())

os.makedirs("data/external/RDD2022", exist_ok=True)
os.makedirs("data/processed/images/train", exist_ok=True)
os.makedirs("data/processed/images/val", exist_ok=True)
os.makedirs("data/processed/labels/train", exist_ok=True)
os.makedirs("data/processed/labels/val", exist_ok=True)
os.makedirs("artifacts/checkpoints", exist_ok=True)
os.makedirs("experiments/metrics", exist_ok=True)
os.makedirs("experiments/visualizations", exist_ok=True)
print("Environment initialized.")

In [ ]:
# 4. Data Preparation: Convert RDD2022 VOC XML to Ultralytics YOLO txt Format
from ml.data.convert_annotations import convert_voc_dir_to_yolo
from ml.data.audit_dataset import audit_dataset_split

print("Converting dataset annotations to YOLO format with official 4-class mapping...")
# When running with real downloaded data, specify actual country directories
# e.g., convert_voc_dir_to_yolo('data/external/RDD2022/Japan/annotations/xmls', 'data/external/RDD2022/Japan/images', 'data/processed/labels/train')
print("Conversion module ready.")

In [ ]:
# 5. Experiment 1: International Baseline Training (Ultralytics YOLO26n / YOLO11n)
from ultralytics import YOLO

print("Starting Experiment 1: Training baseline model on RDD2022 multi-country data...")
model = YOLO("yolo11n.pt")  # or yolo26n.pt

# Training command:
# results = model.train(
#     data="configs/dataset.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     patience=10,
#     project="artifacts/checkpoints",
#     name="exp1_baseline",
#     seed=42,
# )
# model.export(format="onnx")
print("Experiment 1 baseline pipeline defined.")

In [ ]:
# 6. Experiments 2, 3, & 4: Cross-Country LOCO, Zero-Shot, and Data Efficiency Curve
from ml.evaluation.cross_domain import run_experiment_2_loco, run_experiment_4_data_efficiency

loco_results = run_experiment_2_loco()
efficiency_curve = run_experiment_4_data_efficiency()
print("LOCO Cross-Validation and Efficiency curves executed:")
print("LOCO Results:", loco_results)
print("Efficiency Curve:", efficiency_curve)

In [ ]:
# 7. Experiment 5: Inference Latency and FPS Benchmark
from ml.evaluation.benchmark_speed import benchmark_model_speed

speed_metrics = benchmark_model_speed(model_type="yolo", num_warmup=10, num_runs=50)
print("Speed Metrics:", speed_metrics)

In [ ]:
# 8. Generate All 8 Matplotlib Visualizations
from ml.evaluation.cross_domain import generate_all_evaluation_plots

plots = generate_all_evaluation_plots(output_dir="experiments/visualizations")
print(f"Generated {len(plots)} publication figures:")
for p in plots:
    print(" - ", p)

In [ ]:
# 9. Package Colab Artifacts for Download
!zip -r roadsight_colab_artifacts.zip artifacts/ configs/ experiments/